In [8]:
import pandas as pd

In [9]:
df = pd.read_csv("data/cleaned_data.csv")
df.head(1)

,id,title,locations_derived,date_posted,url,ai_education,lats_derived,lngs_derived,ai_experience_level,ai_work_arrangement,job_description
0,2408414159,Associate - Application Engineer,"['Gurugram, Haryana, India']",2026-10-10T15:50:19,https://fa-ewjt-saasfaprod1.fa.ocs.oraclecloud...,"['bachelor degree', 'postgraduate degree']",[28.4646148],[77.0299194],0-2,Hybrid,Associate Application Engineers assist in the ...


In [10]:
df.shape

(999, 11)

In [11]:
import re
from bs4 import BeautifulSoup

def clean_text(text):
    if not isinstance(text, str):
        return text
        
    text = text.lower()
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r"""[!"$%()*,:;<=>?@\[\\\]^_`{|}~.]""", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

ModuleNotFoundError: No module named 'bs4'

In [ ]:
df['tags'] = df['job_description'].apply(clean_text)

In [ ]:
df['tags'][0]

"associate application engineers assist in the development and maintenance of software applications duties include coding debugging and testing applications under the guidance of senior engineers you will work to understand user requirements and contribute to the development of application features basic programming skills and a willingness to learn are necessary responsibilities assist in developing and maintaining applications support application configuration and customization document application processes and workflows perform basic troubleshooting of application issues collaborate with senior application engineers on projects qualifications bachelor's/master's in engineering 0-2 years software development debugging testing application configuration customization troubleshooting genesys cloud cx ccaas google gecx agent assist incident detection triage programming requires a bachelor's or master's degree in engineering with 0-2 years of experience candidates should possess basic pr

In [ ]:
df.shape

(999, 12)

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return text

    return ' '.join(
        word for word in text.split()
        if word not in stop_words
    )

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df['stop_word_removed'] = df['tags'].apply(remove_stopwords)

In [ ]:
df['stop_word_removed'][0]

"associate application engineers assist development maintenance software applications duties include coding debugging testing applications guidance senior engineers work understand user requirements contribute development application features basic programming skills willingness learn necessary responsibilities assist developing maintaining applications support application configuration customization document application processes workflows perform basic troubleshooting application issues collaborate senior application engineers projects qualifications bachelor's/master's engineering 0-2 years software development debugging testing application configuration customization troubleshooting genesys cloud cx ccaas google gecx agent assist incident detection triage programming requires bachelor's master's degree engineering 0-2 years experience candidates possess basic programming skills willingness learn"

In [ ]:
df.shape

(999, 13)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

model_name = "AchrafSoltani/jobbert-ner-sonnet-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def extract_skills(text):
    if not isinstance(text, str) or not text.strip():
        return []

    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunk_size = 450
    skills = []

    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        entities = ner(chunk_text)

        for ent in entities:
            if ent["entity_group"].upper() == "SKILL":
                skill = ent["word"].strip()

                # Remove standalone WordPiece fragments
                if skill.startswith("##"):
                    continue

                # Remove empty or invalid fragments
                if not skill or skill in {"s", "ing", "ation", "-"}:
                    continue

                if skill not in skills:
                    skills.append(skill)

    return skills

In [6]:
#text = df["description_text"].iloc[444]
text = """
'Job Summary:\n\nResponsible for building high-quality, innovative and fully performing software\nin compliance with coding standards and technical design. Design, modify,\ndevelop, write and implement software programming applications. Support and/or\ninstall software applications. Key participant in the testing process through\ntest review and analysis, test witnessing and certification of software.\n\n\xa0\n\nKey Responsibilities:\n\nDevelop software solutions by studying information needs; conferring with users;\nstudying systems flow, data usage and work processes; investigating problem\nareas; following the software development lifecycle; Document and demonstrate\nsolutions; Develops flow charts, layouts and documentation Determine feasibility\nby evaluating analysis, problem definition, requirements, solution development\nand proposed solutions; Understand business needs and know how to create the\ntools to manage them Prepare and install solutions by determining and designing\nsystem specifications, standards and programming Recommend state-of-the-art\ndevelopment tools, programming techniques and computing equipment; participate\nin educational opportunities; read professional publications; maintain personal\nnetworks; participate in professional organizations; remain passionate about\ngreat technologies, especially open source Provide information by collecting,\nanalyzing, and summarizing development and issues while protecting IT assets by\nkeeping information confidential; Improve applications by conducting systems\nanalysis recommending changes in policies and procedures Define applications and\ntheir interfaces, allocate responsibilities to applications, understand solution\ndeployment, and communicate requirements for interactions with solution context,\ndefine Nonfunctional Requirements (NFRs) Understands multiple architectures and\nhow to apply architecture to solutions; understands programming and testing\nstandards; understands industry standards for traditional and agile development\nProvide oversight and foster Built-In Quality and Team and Technical Agility;\nAdopt new mindsets and habits in how people approach their work while supporting\ndecentralized decision making.\n\n\nRESPONSIBILITIES\n\nCompetencies: Security & Compliance Principles - Applies standards, tools, and\nbest practices to embed security, privacy, and compliance into the\ndesign/build/test/operate lifecycle for products, services, apps, systems,\nsoftware, and configurations—balancing protection, efficiency, and cost.\nProgramming Principles - Applies programming languages, frameworks, and patterns\nto design, write, configure, test, and maintain software/solutions/systems that\nare efficient, secure, scalable, and reliable. Data Principles - Governs,\nmodels, secures, implements, and observes data flows to ensure integrity,\nquality, and compliance—enabling trusted, scalable, cost-conscious data use.\nModern Development Practices - Applies modern engineering practices and\ntools—such as Agile/DevSecOps, CI/CD, automated testing, and infrastructure as\ncode—to accelerate delivery, improve quality, and reduce risk across the SDLC.\nSolution Design - Translate business requirements into integrated designs,\narchitectures, patterns, and system interactions that deliver customer value and\nalign with enterprise standards and subject-matter platforms. Demonstrating\nMastery - Maintains essential knowledge and proficiency in relevant domains,\ntools, technologies, methodologies, or frameworks through targeted credentials\nand rigorous proficiency, future-proofing organizational skills against\nstrategic needs. Strategic and Innovative Thinking - Evaluates business and\ntechnology trends, anticipates future needs, develops creative approaches, and\nframes innovations to shape strategy and create durable value with cost-aware\ninnovation. Technical Passion & Drive - Models curiosity and excitement for\ntechnology by self-initiating continuous development, experimenting with\nemerging technologies, and identifying insertion opportunities that accelerate\nbusiness performance. Driving Effective Outcomes - Takes ownership, acts with\nurgency, and initiates action to turn goals into clear plans, decisions,\nguardrails, and cadences while navigating ambiguity and change to drive momentum\nand deliver consistent results. Engaging with Impact - Communicates with clarity\nand purpose to align stakeholders, foster collaboration, build trust, and\ninfluence coordinated action across teams and functions to accelerate outcomes.\nValues Differences - Recognizing the value that different perspectives and\ncultures bring to an organization. Ensuring Customer Success - Embraces a\ncustomer-first mindset to deliver outcomes by linking customer needs and\nbusiness priorities to aligned solutions, delivery, adoption, satisfaction, and\nrealized value through sustained engagement that builds partnership and trust.\nEducation, Licenses, Certifications: College, university, or equivalent degree\nin Computer Science, Engineering, or related subject, or relevant equivalent\nexperience required. This position may require licensing for compliance with\nexport controls or sanctions regulations. Experience: Experience working as a\nsoftware engineer with the following knowledge and experiences are preferred: -\nWorking in Agile environments; - Fundamental IT technical skill sets; - Taking a\nsystem from coping requirements through actual launch of the system; -\nCommunicating with users, other technical teams and management to collect\nrequirements, identify tasks, provide estimates and meet production deadlines; -\nProfessional software engineering best practices for the full software\ndevelopment life cycle, including coding standards, code reviews, source control\nmanagement, build processes, testing and operations.\n\n\nQUALIFICATIONS\n\nRole Summary: The AI Engineer will design, build, validate, and deploy\nAI-enabled solutions that help Cummins teams convert complex operating, product,\nengineering, and business data into practical recommendations and intelligent\nworkflows. This role will work closely with product owners, data scientists,\ndata engineers, architects, and business stakeholders to deliver reusable AI\ncapabilities across the Analytics and AI portfolio.\n\nSkills\n\n * Strong analytical thinking and problem-solving skills with the ability to\n   design AI solutions for complex business and technical challenges.\n * Ability to bridge software engineering fundamentals with modern AI/GenAI\n   technologies to deliver production-ready solutions.\n * Effective communication skills with the ability to explain AI concepts,\n   architecture decisions, and technical recommendations to both technical and\n   non-technical stakeholders.\n * Customer-focused mindset with the ability to understand business problems and\n   translate them into scalable AI applications.\n\n * Strong technical leadership, mentoring, and decision-making capabilities in\n   fast-paced and evolving technology environments.\n   \n   \xa0\n\nTechnical Skills\n\n * AI Engineering & Generative AI\n   * Strong experience building, deploying, and maintaining AI-powered\n     applications and intelligent agents.\n   * Expertise in Large Language Model (LLM) application development and\n     Generative AI solution design.\n   * Experience developing agentic AI workflows, multi-agent systems, and AI\n     orchestration frameworks.\n   * Knowledge of prompt engineering, agent evaluation, guardrails, grounding\n     techniques, and AI application observability.\n   * Familiarity with Retrieval-Augmented Generation (RAG), semantic search,\n     vector databases, and knowledge management architectures.\n * Open Source AI Frameworks\n   * Hands-on experience with modern AI orchestration frameworks including\n     LangGraph, LangChain, LlamaIndex Hugging Face, Open-Source LLM ecosystems\n   * Experience integrating both commercial and open-source foundation models.\n   * Knowledge of model serving, inference optimization, and AI workflow\n     orchestration patterns.\n * Software Engineering & Application Development\n   * Strong proficiency in Python with experience building production-grade\n     applications and APIs.\n   * Experience with modern software engineering practices including\n     object-oriented design, design patterns, testing, and secure coding\n     principles.\n   * Experience developing REST APIs, microservices, and event-driven\n     architectures.\n   * Familiarity with front-end technologies such as React, Angular, or similar\n     frameworks for AI application development.\n   * Understanding of distributed systems and scalable application architecture.\n * Data Engineering & AI Platforms\n   * Experience working with Azure Databricks, Spark, and modern data platforms\n     supporting AI workloads.\n   * Strong understanding of data pipelines, feature engineering, and data\n     preparation for AI applications.\n   * Experience integrating enterprise data sources, structured and unstructured\n     datasets, and knowledge repositories.\n   * Familiarity with vector databases and embedding-based retrieval\n     architectures.\n   * Understanding of modern data lake, warehouse, and lakehouse architectures.\n * Cloud, DevOps & MLOps\n   * Strong experience with Azure cloud services supporting AI workloads,\n     including:\n     * Azure AI Services\n     * Azure OpenAI\n     * Azure Databricks\n     * Azure Functions\n     * Azure Kubernetes Service (AKS)\n     * Azure Storage and Data Services\n   * Experience with Docker, Kubernetes, and cloud-native application\n     deployment.\n   * Familiarity with MLOps and LLMOps practices including model lifecycle\n     management, monitoring, and continuous deployment.\n   * Experience implementing CI/CD pipelines using Git, Azure DevOps, Jenkins,\n     or equivalent platforms.\n   * Knowledge of AI governance, security, responsible AI, and compliance\n     practices.\n * AI Solution Architecture\n   * Experience designing enterprise-scale AI platforms and reusable AI\n     services.\n   * Ability to evaluate AI technologies and recommend architecture patterns\n     aligned with business objectives.\n   * Knowledge of AI system performance optimization, scalability, reliability,\n     and operational excellence.\n   \n   * Experience establishing engineering standards, reusable frameworks, and\n     best practices for AI solution delivery.\n     \n     \xa0\n\nExperience\n\n * Minimum 8+ years of hands-on experience in Software Engineering, Full Stack\n   Development, Platform Engineering, Data Engineering, or related technical\n   fields.\n * Minimum 3+ years of experience designing, developing, or deploying AI,\n   Machine Learning, or Generative AI solutions.\n * Demonstrated success transitioning enterprise applications from traditional\n   software architectures to AI-enabled solutions.\n * Proven experience building and deploying scalable production applications in\n   cloud environments.\n * Experience integrating AI capabilities into business applications, workflows,\n   and enterprise platforms.\n * Experience working with modern AI frameworks, LLMs, and agent-based\n   application architectures.\n * Experience operating within Agile software development environments.\n * Experience collaborating with product managers, data scientists, architects,\n   and business stakeholders to deliver innovative solutions.\n * Proven ability to lead technical initiatives and mentor engineering teams.\n\n\xa0\n\nNice to Have\n\n * Experience fine-tuning, evaluating, or optimizing open-source models.\n * Experience with Databricks Mosaic AI, MLflow, or other enterprise AI\n   development platforms.\n * Knowledge of advanced AI techniques such as multi-agent systems, AI workflow\n   orchestration, and autonomous agents.\n * Experience with graph databases such as Neo4j and knowledge graph\n   implementations.\n * Familiarity with Deep Learning frameworks including PyTorch and TensorFlow.\n * Experience building enterprise Copilots and conversational AI solutions.\n * Azure AI Engineer Associate, Azure Developer Associate, Databricks, or\n   equivalent cloud certifications.\n * Understanding of Responsible AI, AI governance, and enterprise compliance\n   requirements.\n\n\xa0\n\nCompetencies\n\n * Artificial Intelligence & Generative AI\n * Software Engineering Excellence\n * Cloud & AI Platform Architecture\n * Problem Solving\n * Communicates Effectively\n\n\nCummins is an equal opportunity employer. Our policy is to provide equal\nemployment opportunities to all qualified persons without regard to race, sex,\ncolor, disability, national origin, age, religion, union affiliation, sexual\norientation, veteran status, citizenship, gender identity, or other status\nprotected by law.'
"""

skills = extract_skills(text)

print(skills)

NameError: name 'extract_skills' is not defined

In [8]:
import ast
import re
import string

def clean_skills(skills_text):
    # Handle missing or invalid input
    if not isinstance(skills_text, str) or not skills_text.strip():
        return []

    # Convert string representation into a Python list
    try:
        skills = ast.literal_eval(skills_text)
    except (ValueError, SyntaxError):
        return []

    if not isinstance(skills, list):
        return []

    cleaned_skills = []

    for skill in skills:
        if not isinstance(skill, str):
            continue

        # Remove all whitespace
        skill = re.sub(r"\s+", "", skill)

        # Skip empty skills
        if not skill:
            continue

        # Remove skills containing only punctuation
        if all(char in string.punctuation for char in skill):
            continue

        # Remove unwanted prefix punctuation
        skill = re.sub(r"^[^\w]+", "", skill)

        # Remove unwanted suffix punctuation, preserving + and #
        skill = re.sub(r"[^\w+#]+$", "", skill)

        if not skill:
            continue

        # Remove standalone numbers
        if re.fullmatch(r"\d+", skill):
            continue

        # Remove single alphabet characters except valid C and R language variants
        if re.fullmatch(r"[a-zA-Z]", skill) and skill.lower() not in {"c", "r"}:
            continue

        cleaned_skills.append(skill)

    return cleaned_skills


# Apply the function
df["cleaned_skills"] = df["extracted_skills"].apply(clean_skills)

KeyError: 'extracted_skills'

In [125]:
skill_counts = (
    df["cleaned_skills"]
    .explode()                     # Expand each list into individual skills
    .dropna()                      # Remove missing values
    .str.strip()                   # Remove leading/trailing spaces
    .str.lower()                   # Convert to lowercase
    .loc[lambda s: s.ne("")]       # Remove empty strings
    .value_counts()
    #.sort_index(ascending=True)    # Sort alphabetically
)

pd.set_option("display.max_rows", None)

print(len(skill_counts))
display(skill_counts)

8687


cleaned_skills
management                                                                                            275
communication                                                                                         173
communicationskills                                                                                   160
stakeholdermanagement                                                                                 146
attentiondetail                                                                                       123
problem-solving                                                                                       113
skills                                                                                                112
analysis                                                                                              110
analyticalskills                                                                                      105
problemsolving                 

In [126]:
df['cleaned_skills'][453]

['problem-solving',
 'verbalcommunicationskills',
 'computerliteracy',
 'customersupport',
 'voiceprocessoperations',
 'basiccomputeroperationssoftware',
 'verbalcommunicationcapabilities',
 'problem-solvingcommunicationskills',
 'conflictresolution',
 'customerservice',
 'complainthandling']

In [127]:
additional_skill_mapping = {
    # Analytical and problem-solving skills
    "analytical": "analyticalskills",
    "analyticskills": "analyticalskills",
    "analyticalthinking": "analyticalskills",
    "analyticalthinkingskills": "analyticalskills",
    "analyticalability": "analyticalskills",
    "analyticalabilities": "analyticalskills",
    "analyticalcapability": "analyticalskills",
    "analyticalcapabilities": "analyticalskills",
    "problemsolving": "problemsolvingskills",
    "problem-solving": "problemsolvingskills",
    "problem-solvingabilities": "problemsolvingskills",
    "problem-solvingcapabilities": "problemsolvingskills",
    "criticalthinking": "criticalthinkingskills",
    "criticalthinkingskills": "criticalthinkingskills",
    "decisionmaking": "decisionmakingskills",
    "decisionmakingskills": "decisionmakingskills",

    # Communication and interpersonal skills
    "communication": "communicationskills",
    "communications": "communicationskills",
    "communicationskill": "communicationskills",
    "verbalcommunication": "communicationskills",
    "verbalcommunicationskills": "communicationskills",
    "writtencommunication": "communicationskills",
    "writtencommunicationskills": "communicationskills",
    "interpersonalskills": "interpersonalskills",
    "activelistening": "activelistening",
    "activelisteningskills": "activelistening",
    "conflictresolution": "conflictresolution",
    "customerservice": "customerservice",
    "customersupport": "customersupport",
    "complainthandling": "complainthandling",

    # Attention to detail
    "attentiontodetail": "attentiontodetail",
    "attentiondetail": "attentiontodetail",
    "attentiontodetailskills": "attentiontodetail",
    "attentiontodetailaccuracy": "attentiontodetail",

    # Leadership and teamwork
    "leadership": "leadershipskills",
    "leadershipskill": "leadershipskills",
    "teamleadership": "leadershipskills",
    "teamwork": "teamwork",
    "teamworkskills": "teamwork",
    "collaboration": "collaboration",
    "collaborationskills": "collaboration",
    "adaptability": "adaptability",
    "flexibility": "adaptability",
    "timemanagement": "timemanagement",
    "organizational": "organizationalskills",
    "organizationalskills": "organizationalskills",

    # Programming and software development
    "py": "python",
    "pythonprogramming": "python",
    "java-script": "javascript",
    "js": "javascript",
    "nodejs": "node.js",
    "reactjs": "react",
    "angularjs": "angular",
    "dotnet": ".net",
    "csharp": "c#",
    "cplusplus": "c++",
    "restfulapi": "restapi",
    "restapis": "restapi",
    "apis": "api",
    "microservicesdevelopment": "microservices",
    "softwaredevelopmentdebuggingtesting": "softwaredevelopment",

    # Databases and data
    "postgres": "postgresql",
    "postgre": "postgresql",
    "mssql": "sqlserver",
    "sqlserver": "sqlserver",
    "structuredquerylanguage": "sql",
    "powerbi": "powerbi",
    "power-bi": "powerbi",
    "ms-excel": "msexcel",
    "microsoftexcel": "msexcel",
    "excel": "msexcel",
    "tableau": "tableau",
    "dataanalytics": "dataanalytics",
    "dataanalysis": "dataanalytics",
    "dataanalyst": "dataanalytics",
    "datavisualization": "datavisualization",
    "datamodelling": "datamodeling",
    "datamodeling": "datamodeling",

    # AI and machine learning
    "ml": "machinelearning",
    "machine-learning": "machinelearning",
    "machinelearning": "machinelearning",
    "ai": "artificialintelligence",
    "artificialintelligence": "artificialintelligence",
    "nlp": "naturallanguageprocessing",
    "naturallanguageprocessing": "naturallanguageprocessing",
    "deeplearning": "deeplearning",
    "generativeai": "generativeai",
    "genai": "generativeai",
    "computervision": "computervision",

    # Cloud and DevOps
    "aws": "amazonwebservices",
    "amazonwebservices": "amazonwebservices",
    "gcp": "googlecloudplatform",
    "googlecloud": "googlecloudplatform",
    "googlecloudplatform": "googlecloudplatform",
    "azurecloudplatforms": "azure",
    "azuredevops": "azuredevops",
    "k8s": "kubernetes",
    "kubernetes": "kubernetes",
    "ci/cd": "cicd",
    "cicd": "cicd",
    "terraform": "terraform",
    "ansible": "ansible",
    "docker": "docker",
    "jenkins": "jenkins",

    # Accounting and finance
    "accountspayable": "accountspayable",
    "accountspayables": "accountspayable",
    "accountsreceivable": "accountsreceivable",
    "accountsreceivables": "accountsreceivable",
    "bankreconciliation": "bankreconciliation",
    "bankreconciliations": "bankreconciliation",
    "balancesheetanalysis": "balancesheetanalysis",
    "financialreporting": "financialreporting",
    "financialanalysis": "financialanalysis",
    "bookkeepingpractices": "bookkeeping",

    # Sales and business
    "sale": "sales",
    "sales": "sales",
    "businessdevelopment": "businessdevelopment",
    "businessdevelopmentskills": "businessdevelopment",
    "accountmanagement": "accountmanagement",
    "customerrelationshipmanagement": "crm",
    "crmsoftware": "crm",
    "agilemethodology": "agile",
    "agilemethodologies": "agile",
    "scrummethodology": "scrum",
    "scrummethodologies": "scrum",
}

In [103]:
import re

skill_mapping = {
    "excel": "msexcel",
    "microsoftexcel": "msexcel",
    "mswordexcel": "msexcel",
    "communication": "communicationskills",
    "communications": "communicationskills",
    "communicationskill": "communicationskills",
    "problem-solving": "problemsolving",
    "problem-solvingskills": "problemsolving",
    "git": "github",
    "pipelines": "pipeline",
    "ml": "machinelearning",
    "/ml": "machinelearning",
    "ai": "artificialintelligence",
    "&development": "development",
    "accesscontrols": "accesscontrol",
    "-sales": "sales",
    "sale": "sales",
    "mysql": "sql",
    "mssql": "sql",
    "js": "javascript",
    "java-script": "javascript",
    "postgres": "postgresql",
    "aws": "amazonwebservices",
    "gcp": "googlecloudplatform",
    "googlecloud": "googlecloudplatform",
    "cloudservices": "cloud",
    "-cloud": "cloud",
    "analysis": "analyticalskills",
    "leadership": "leadershipskills",
    "teamleadership": "leadershipskills"
}

def normalize_skills(skills):
    # Handle missing values or unexpected data types
    if not isinstance(skills, list):
        return []

    normalized_skills = []

    for skill in skills:
        if not isinstance(skill, str):
            continue

        skill = skill.lower().strip()

        # Apply longer mappings first
        for variation, standard in sorted(
            skill_mapping.items(),
            key=lambda item: len(item[0]),
            reverse=True
        ):
            pattern = r"(?<!\w)" + re.escape(variation) + r"(?!\w)"
            skill = re.sub(pattern, standard, skill)

        if skill:
            normalized_skills.append(skill)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(normalized_skills))


df["normalized_skills"] = df["cleaned_skills"].apply(normalize_skills)

In [104]:
df['normalized_skills'][0]

['coding',
 'debuggingtesting',
 'basicprogrammingskills',
 'troubleshooting',
 'softwaredevelopmentdebuggingtesting',
 'applicationconfiguration',
 'cloudcxccaas',
 'googlegecxagent',
 'incidentdetection',
 'triageprogramming',
 'skills']

In [128]:
import pandas as pd
pd.set_option("display.max_rows", None)
word_counts = (
    df["normalized_skills"]
    .dropna()
    #.str.lower()
    #.str.split()
    .explode()
    .value_counts()
    #.sort_index(ascending=True)
)

print(len(word_counts))
display(word_counts)

981


normalized_skills
portfoliomanagement customernegotiation compliance negotiationskills                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [106]:
df.head(1)

,id,title,locations_derived,date_posted,url,ai_education,lats_derived,lngs_derived,ai_experience_level,ai_work_arrangement,job_description,tags,extracted_skills,cleaned_skills,normalized_skills
0,2408414159,Associate - Application Engineer,"['Gurugram, Haryana, India']",2026-10-10T15:50:19,https://fa-ewjt-saasfaprod1.fa.ocs.oraclecloud...,"['bachelor degree', 'postgraduate degree']",[28.4646148],[77.0299194],0-2,Hybrid,Associate Application Engineers assist in the ...,associate application engineers assist develop...,"['coding', 'debugging testing', 'basic program...","[coding, debuggingtesting, basicprogrammingski...","[coding, debuggingtesting, basicprogrammingski..."


In [107]:
df['normalized_skills'][0]

['coding',
 'debuggingtesting',
 'basicprogrammingskills',
 'troubleshooting',
 'softwaredevelopmentdebuggingtesting',
 'applicationconfiguration',
 'cloudcxccaas',
 'googlegecxagent',
 'incidentdetection',
 'triageprogramming',
 'skills']

In [108]:
print(type(df['normalized_skills'][0]))

<class 'list'>


In [113]:
df['normalized_skills'] = df['normalized_skills'].apply(lambda x:" ".join(x))

In [119]:
df['normalized_skills'][0]

'coding debuggingtesting basicprogrammingskills troubleshooting softwaredevelopmentdebuggingtesting applicationconfiguration cloudcxccaas googlegecxagent incidentdetection triageprogramming skills'

In [114]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(binary=True, stop_words='english')

In [115]:
vectors = cv.fit_transform(df['normalized_skills']).toarray() #numpyarray

In [116]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(999, 8624))

In [117]:
vectors.shape

(999, 8624)

In [120]:
cv.get_feature_names_out()

array(['12', '22nm16nm', '2dauthoringsolidworkselectrical3', ...,
       'zonecommunicationdesign', 'zoominfolinkedin',
       'ürelationshipmanagementskill'], shape=(8624,), dtype=object)

In [121]:
from sklearn.metrics.pairwise import cosine_similarity 

In [122]:
cosine_similarity(vectors).shape

(999, 999)

In [123]:
similarity = cosine_similarity(vectors)
similarity = similarity.astype("float32")
similarity

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.05504819],
       [0.        , 1.        , 0.13944334, ..., 0.04546629, 0.03849002,
        0.        ],
       [0.        , 0.13944334, 1.        , ..., 0.04075696, 0.        ,
        0.048795  ],
       ...,
       [0.        , 0.04546629, 0.04075696, ..., 1.        , 0.03937496,
        0.0278423 ],
       [0.        , 0.03849002, 0.        , ..., 0.03937496, 1.        ,
        0.04714045],
       [0.05504819, 0.        , 0.048795  , ..., 0.0278423 , 0.04714045,
        1.        ]], shape=(999, 999), dtype=float32)

In [124]:
similarity[0].shape

(999,)